# Document Length Distribution Comparison

Loads multiple JSONL files and overlays character-length distributions.
Each file entry specifies its own text field, so datasets with different schemas can be compared side-by-side.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

## Configuration

Each entry in `FILES` is a `(path, label, field)` tuple:
- **path** – path to the JSONL file
- **label** – display name used in the legend
- **field** – the JSON key whose value will be measured (e.g. `"final_text"`, `"body"`, `"content"`)

Adjust `BIN_COUNT`, `LOG_SCALE`, and `ALPHA` as needed.

In [ ]:
FILES = [
    ("./final/DB-BIO/Original/test.jsonl", "Original", "final_text"),
    ("./final/DB-BIO/reflexion/reflexion_wiki_2026-07-24_01-39-38_model_gpt-4o_iters_5_seed_42_pak_1_noutil_False_cot_False_mem_3_pthr_10.jsonl", "RUPTA", "final_text"),
    # ("./final/DB-BIO/Aux_Linking_Fact_Distorter/aux_linking_fact_distortion_wiki_2026-07-23_23-06-06_model_gpt-4o_iters_5_seed_42_pak_1_noutil_False_cot_False_mem_3_tau_70_sorted.jsonl", "ALTA-FD", "final_text"),
    # Add more entries here, e.g.:
    # ("dataset_c.jsonl", "Dataset C", "content"),
]

BIN_COUNT = 40     # number of histogram bins
LOG_SCALE = False  # set True to use log scale on the x-axis
ALPHA     = 0.3   # transparency for overlapping fills (0-1)

## Load data

In [ ]:
def load_lengths(path: str, field: str) -> list[int]:
    """Return character lengths of `field` for every record in a JSONL file."""
    lengths = []
    missing = 0
    with open(path, encoding="utf-8") as f:
        for lineno, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"  [warn] {path}:{lineno} - JSON parse error: {e}")
                continue
            value = record.get(field)
            if value is None:
                missing += 1
            else:
                lengths.append(len(str(value)))
    if missing:
        print(f"  [warn] {path}: {missing} record(s) missing '{field}' key")
    return lengths


datasets = []
for path, label, field in FILES:
    p = Path(path)
    if not p.exists():
        print(f"[ERROR] File not found: {path}")
        continue
    lengths = load_lengths(path, field)
    datasets.append({"label": label, "field": field, "lengths": np.array(lengths)})
    print(f"{label} (field='{field}'): {len(lengths):,} documents loaded from '{path}'")

# Build a long-form DataFrame for seaborn
df = pd.concat(
    [
        pd.DataFrame({"length": ds["lengths"], "dataset": ds["label"]})
        for ds in datasets
        if len(ds["lengths"]) > 0
    ],
    ignore_index=True,
)

## Summary statistics

In [ ]:
print(f"{'Dataset':<20} {'Field':<16} {'Count':>10} {'Min':>10} {'Median':>10} {'Mean':>10} {'P95':>10} {'Max':>12}")
print("-" * 100)
for ds in datasets:
    arr = ds["lengths"]
    if len(arr) == 0:
        print(f"{ds['label']:<20}  (no data)")
        continue
    print(
        f"{ds['label']:<20}"
        f"{ds['field']:<16}"
        f"{len(arr):>10,}"
        f"{int(arr.min()):>10,}"
        f"{int(np.median(arr)):>10,}"
        f"{int(arr.mean()):>10,}"
        f"{int(np.percentile(arr, 95)):>10,}"
        f"{int(arr.max()):>12,}"
    )

## Overlaid KDE + histogram

In [ ]:
if df.empty:
    print("No datasets loaded. Check the file paths in the Configuration cell.")
else:
    sns.set_theme(style="whitegrid", context="notebook", font_scale=1.15)
    palette = sns.color_palette("tab10", n_colors=len(datasets))

    all_lengths = df["length"].values
    if LOG_SCALE:
        min_val = max(all_lengths.min(), 1)
        bins = np.logspace(np.log10(min_val), np.log10(all_lengths.max()), BIN_COUNT + 1)
    else:
        bins = np.linspace(all_lengths.min(), all_lengths.max(), BIN_COUNT + 1)

    fig, ax = plt.subplots(figsize=(13, 5.5))

    for i, ds in enumerate(datasets):
        arr = ds["lengths"]
        if len(arr) == 0:
            continue
        color = palette[i]
        label = f"{ds['label']}  (n={len(arr):,}, field='{ds['field']}')"

        # Histogram bars
        ax.hist(
            arr,
            bins=bins,
            density=True,
            alpha=ALPHA,
            color=color,
            edgecolor="none",
        )
        # # Smooth KDE line on top
        # sns.kdeplot(
        #     arr,
        #     ax=ax,
        #     color=color,
        #     linewidth=2.2,
        #     label=label,
        #     log_scale=LOG_SCALE,
        #     warn_singular=False,
        # )

    if LOG_SCALE:
        ax.set_xscale("log")

    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    ax.set_xlabel("Document length (characters)", labelpad=8)
    ax.set_ylabel("Density", labelpad=8)
    ax.set_title("Document length distribution by dataset", pad=14, fontweight="bold")
    ax.legend(framealpha=0.9, edgecolor="#cccccc")
    sns.despine(left=False)
    fig.tight_layout()
    plt.show()

## CDF overlay (optional)

Cumulative distribution functions make it easy to compare percentiles directly.

In [ ]:
if not df.empty:
    sns.set_theme(style="whitegrid", context="notebook", font_scale=1.15)

    fig, ax = plt.subplots(figsize=(13, 5.5))

    for i, ds in enumerate(datasets):
        arr = np.sort(ds["lengths"])
        if len(arr) == 0:
            continue
        cdf = np.arange(1, len(arr) + 1) / len(arr)
        label = f"{ds['label']}  (n={len(arr):,}, field='{ds['field']}')"
        ax.plot(arr, cdf, color=palette[i], linewidth=2.2, label=label)
        # Shade under the curve for a bit more visual richness
        ax.fill_between(arr, cdf, alpha=0.08, color=palette[i])

    # Reference percentile lines
    for p, ls in zip([0.50, 0.90, 0.95, 0.99], ["--", ":", ":", ":"]):
        ax.axhline(p, color="#888888", linestyle=ls, linewidth=0.9, alpha=0.8)
        ax.text(
            ax.get_xlim()[1] * 0.01, p + 0.012,
            f"p{int(p * 100)}",
            fontsize=8.5, color="#666666", va="bottom",
        )

    if LOG_SCALE:
        ax.set_xscale("log")

    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    ax.set_xlabel("Document length (characters)", labelpad=8)
    ax.set_ylabel("Cumulative fraction", labelpad=8)
    ax.set_title("CDF of document length by dataset", pad=14, fontweight="bold")
    ax.legend(framealpha=0.9, edgecolor="#cccccc")
    sns.despine(left=False)
    fig.tight_layout()
    plt.show()